In [ ]:
import json
import os
from collections import defaultdict

import numpy as np
import pandas as pd

from models.SiamABC.tracker.tracker_setup import get_tracker
from models.SiamRAM import SiamRAMTracker
from utils.hydra import load_hydra_config_from_path
from vis.test_model import run_inference

with open("/home/moha/AIC-4/data_competition/metadata/contestant_manifest.json", "r") as f:
    manifest = json.load(f)
test_public_lb = manifest["public_lb"]
manifest.keys()

dict_keys(['train', 'public_lb'])

In [ ]:





video_paths = [
    # "/home/moha/AIC-4/data_competition/dataset2/Gull2/Gull2_24.mp4",
    # "/home/moha/AIC-4/data_competition/dataset3/car16_3/car16_3_24.mp4",
    # "/home/moha/AIC-4/data_competition/dataset2/RaceCar1/RaceCar1_24.mp4",
    # "/home/moha/AIC-4/data_competition/dataset5/uav2/uav2_30.mp4",
    # "/home/moha/AIC-4/data_competition/dataset5/uav7/uav7_30.mp4",
    # "/home/moha/AIC-4/data_competition/dataset2/RcCar6/RcCar6_24.mp4",
    # "/home/moha/AIC-4/data_competition/dataset2/Motor1/Motor1_24.mp4",
    # "/home/moha/AIC-4/data_competition/dataset2/MountainBike5/MountainBike5_30.mp4",
    # "/home/moha/AIC-4/data_competition/dataset1/person_3/person_3.mp4",
    # "/home/moha/AIC-4/data_competition/dataset3/truck/truck_30.mp4",
    # "/home/moha/AIC-4/data_competition/dataset3/human3/human3_24.mp4",
    # "/home/moha/AIC-4/data_competition/dataset1/sheeps_2/sheeps_2.mp4",
    # "/home/moha/AIC-4/data_competition/dataset2/Animal3/Animal3_30.mp4",
    # "/home/moha/AIC-4/data_competition/dataset5/person16/person16_96.mp4",
    # "/home/moha/AIC-4/data_competition/dataset3/electric_box/electric_box_24.mp4",
    # "/home/moha/AIC-4/data_competition/dataset2/RcCar4/RcCar4_24.mp4",
    # "/home/moha/AIC-4/data_competition/dataset4/group2/group2_96.mp4",
    # "/home/moha/AIC-4/data_competition/dataset3/jogging2/jogging2_30.mp4",
    # "/home/moha/AIC-4/data_competition/dataset3/car6_2/car6_2_30.mp4",
    # "/home/moha/AIC-4/data_competition/dataset4/person19/person19_96.mp4",
    # "/home/moha/AIC-4/data_competition/dataset3/couple/couple_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/tennis_player1_2/tennis_player1_2_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset5/car4/car4_96.mp4"
]

model_size = "M"
weights_path = "/home/moha/AIC-for submission/SiamRAM/checkpoints/head_epoch_000.pth"
config_path = "../../AIC-4/external/SiamABC/core/config"
config_name = "SiamABC_tracker"

config = load_hydra_config_from_path(config_path=config_path, config_name=config_name)
config["model"]["model_size"] = 'S' if model_size=="S_Tiny" else 'M'
config["tracker"]["N"] =10
config["tracker"]["lr"] = 0.4
config["tracker"]["dynamic_update"] = True 
config["tracker"]["memory_window_size"] = 20
config["tracker"]["dynamic_update_threshold"] = 0.87
config["tracker"]["running_confidence_floor_value"] = 100
config["tracker"]["search_context"] = 2
config["tracker"]["iou_threshold"] = 0.6
config["model"]["_target_"] = "models.SiamABC.model.SiamABC.SiamABCNet"
config["tracker"]["_target_"] = "models.SiamABC.tracker.SiamABC_Tracker.SiamABCTracker"
config["tracker"]["warmup_frames"] = 35
config["tracker"]["warmup_window_size"] = 8

# config["tracker"]["jump_center_shift"] = 0.7
# config["tracker"]["jump_size_ratio"] = 4
# config["tracker"]["jump_score_override"] = 0.99



# config["tracker"]["window_influence"] =  0.45 
# config["tracker"]["penalty_k"] = 0.10


wrapped = get_tracker(config=config, weights_path=weights_path , lambda_tta=0.1 , continuous=False)


tracker = SiamRAMTracker(
        siam_tracker        = wrapped,
        yolo_weights        = "yolo11n.pt",

        conf_threshold      = 0.55,      # score BELOW this starts the entry streak
        reacq_threshold     = 0.7,      # tracker score to exit occlusion (phase 0)
        occ_siam_reacq_threshold = 0.80,
        yolo_conf           = 0.3,
        app_match_threshold = 0.1,      # cosine sim to accept tracker bbox as target
        occ_siam_margin=0.5,
        nudge_alpha         = 0.0,

        # ── NEW: Occlusion entry hysteresis ───────────────────────────────────
        entry_patience      = 1,        # N consecutive bad frames before occlusion

        # ── NEW: Multi-frame candidate collection ─────────────────────────────
        cand_collection_frames = 1,     # YOLO collection frames before final DRM

        # ── NEW: Velocity scoring weight & guard ──────────────────────────────
        drm_lam_cand_vel    = 0.0,     # weight of vel_score in final DRM phase
                                        # set 0 to disable velocity scoring
        vel_score_min_speed = 0.5,      # min EKF speed (px/frame) to use vel score
                                        # if EKF speed < this → vel_score = 0.5 (neutral)

        # ── NEW: Tiny / long-distance object ROI parameters ───────────────────
        # Used automatically when _is_long_distance() returns True.
        long_distance_area_fraction =  0.030,
        tiny_roi_start_expand            = 8.0,
        tiny_yolo_search_expand          = 20.0,
        tiny_search_expand_growth_factor = 1.3,
        tiny_search_expand_growth_every  = 50,
        tiny_search_expand_max           = 40.0,

        # ── DRM (existing params) ─────────────────────────────────────────────
        drm_tau_sim         = 0.6,
        mem_capacity        = 50,
        drm_mmin            = 3,
        drm_capacity        = 20,
        history_decay       = 0.1,
        drm_lam_dist        = 0.0,
        drm_lam_cand_dir    = 0.0,
        drm_lam_time        = 0.0,
        drm_lam_app         = 1,
        drm_lam_iou         = 0.0,
        drm_lam_mot         = 0.0,
        drm_margin          = 0.6,
        drm_skip_threshold  = 23,
        drm_top_k           = 100,
        vel_dir_hard_gate = 0.4,   # |cos| threshold below which score → 0.05
        yolo_filter_class  = False, # filter candidates to target class
        yolo_class_detect_frames   = 5,     # stride 

        # ── History ───────────────────────────────────────────────────────────
        conf_history_len    = 200,
        size_history_len    = 200,
        history_skip_last   = 8,

        # ── ROI expansion (normal objects) ────────────────────────────────────
        roi_start_expand                 = 20,
        yolo_search_expand               = 100,
        search_expand_growth_factor      = 1.4,
        search_expand_growth_every       = 150,
        search_expand_max                = 500.0,

        # ── Misc ──────────────────────────────────────────────────────────────
        velocity_window_average          = 200,
        shrinkage_max_lookback           = 30,
        enter_occlusion_on_loss          = True,
        drm_gamma                        = 0,


    )

start = 1
for i in range(start, start + len(video_paths)):
    video_path = video_paths[i - start]
    ann_path = os.path.join(os.path.dirname(video_path), "annotation.txt")
    output_path = f"outputs/test_5_after/test_{i}.mp4"

    init_bbox = np.loadtxt(ann_path, delimiter=",", dtype=np.float32).tolist()
    if not isinstance(init_bbox[0], (int, float)):
        init_bbox = init_bbox[0]
    print(init_bbox)

    run_inference(
        video_path=video_path,
        initial_bbox=init_bbox,
        tracker=tracker,
        output_path=output_path
    )



{'penalty_k': 0.062, 'window_influence': 0.38, 'lr': 0.4, 'windowing': 'cosine', 'total_stride': 16, 'score_size': 16, 'N': 10, 'dynamic_update': True, 'similarity_score': False, 'stride': 2, 'smooth': False, 'bbox_ratio': 0.5, 'template_bbox_offset': 0.2, 'search_context': 2, 'instance_size': 256, 'template_size': 128, 'memory_window_size': 20, 'dynamic_update_threshold': 0.87, 'running_confidence_floor_value': 100, 'iou_threshold': 0.6, 'warmup_frames': 35, 'warmup_window_size': 8}
[1079.0, 299.0, 32.0, 47.0]
[occlusion entry] frame=1440  loss_cause=out_of_frame  out_of_frame=True  exit_edge=right  entry_streak=1
[occlusion entry] frame=1440  loss_cause=out_of_frame  dynamic_skip=30  entry_streak_skip=1  effective_skip=0  history_len=200
[occ frame 2] phase=collect(1/1)  detections=0  stored=0
[occ frame 3] phase=final_drm  no candidates in any collection frame — resetting
[occ frame 5] phase=collect(1/1)  detections=0  stored=0
[occ frame 6] phase=final_drm  no candidates in any col

KeyboardInterrupt: 

In [ ]:
# 0.768 score on public lb
test_public_lb = manifest["public_lb"]
with open("/home/moha/AIC-4/data_competition/metadata/contestant_manifest.json", "r") as f:
    manifest = json.load(f)
manifest.keys()
test_public_lb = manifest["public_lb"]

# test_pub_2 = {k:v for k , v in manifest["public_lb"].items() if v["dataset"]=="dataset2"}


folder_names = [
    # "uav2",
    # "uav7",
    # "RcCar6",
    "Motor1",
    "MountainBike5",
    "person_3",
    "truck",
    "human3",
    "sheeps_2",
    "Animal3",
    "person16",
    "electric_box",
    "RcCar4",
    "group2",
    "truck",
    "jogging2",
    "car6_2",
    "person19",
    "couple",
    "tennis_player1_2",
    "car4",
]

data_dir = "../../AIC-4/data_competition"
outputs_dir = "outputs/SiamDAM__"
test_public_lb = {
    k: v for k, v in manifest["public_lb"].items()
    if v["dataset"] in [
        "dataset1",
        "dataset2",
        "dataset3",
        "dataset4",
        "dataset5"
    ]
    # and v["seq_name"] in  folder_names
}

for i, (key, value) in enumerate(test_public_lb.items()):
    model_size = "M"
    weights_path = "/home/moha/AIC-for submission/SiamRAM/checkpoints/head_epoch_000.pth"
    # weights_path = "/home/moha/AIC-4/checkpoints/bbox_head/all_datasets_finetuning/part_2/head_epoch_001.pth"
    config_path = "../../AIC-4/external/SiamABC/core/config"
    config_name = "SiamABC_tracker"

    config = load_hydra_config_from_path(config_path=config_path, config_name=config_name)
    config["model"]["model_size"] = 'S' if model_size=="S_Tiny" else 'M'
    config["tracker"]["N"] =10
    config["tracker"]["lr"] = 0.4
    config["tracker"]["dynamic_update"] = True 
    config["tracker"]["memory_window_size"] = 20
    config["tracker"]["dynamic_update_threshold"] = 0.87
    config["tracker"]["running_confidence_floor_value"] = 100
    config["tracker"]["search_context"] = 2
    config["tracker"]["iou_threshold"] = 0.6
    config["model"]["_target_"] = "models.SiamABC.model.SiamABC.SiamABCNet"
    config["tracker"]["_target_"] = "models.SiamABC.tracker.SiamABC_Tracker.SiamABCTracker"
    config["tracker"]["warmup_frames"] = 35
    config["tracker"]["warmup_window_size"] = 8

    # config["tracker"]["jump_center_shift"] = 0.7
    # config["tracker"]["jump_size_ratio"] = 4
    # config["tracker"]["jump_score_override"] = 0.99



    # config["tracker"]["window_influence"] =  0.45 
    # config["tracker"]["penalty_k"] = 0.10


    wrapped = get_tracker(config=config, weights_path=weights_path , lambda_tta=0.1 , continuous=False)
    # siam.all_memory_imgs = deque(maxlen=250)   # was 5000
    # siam.classification_scores = deque(maxlen=250)


    tracker = SiamRAMTracker(
            siam_tracker        = wrapped,
            yolo_weights        = "yolo11n.pt",

            # ── Core thresholds ───────────────────────────────────────────────────
            conf_threshold      = 0.55,      # score BELOW this starts the entry streak
            reacq_threshold     = 0.7,      # tracker score to exit occlusion (phase 0)
            occ_siam_reacq_threshold = 0.80,
            yolo_conf           = 0.3,
            app_match_threshold = 0.1,      # cosine sim to accept tracker bbox as target
            occ_siam_margin=0.5,
            nudge_alpha         = 0.0,

            # ── NEW: Occlusion entry hysteresis ───────────────────────────────────
            entry_patience      = 1,        # N consecutive bad frames before occlusion

            # ── NEW: Multi-frame candidate collection ─────────────────────────────
            cand_collection_frames = 1,     # YOLO collection frames before final DRM

            # ── NEW: Velocity scoring weight & guard ──────────────────────────────
            drm_lam_cand_vel    = 0.0,     # weight of vel_score in final DRM phase
                                            # set 0 to disable velocity scoring
            vel_score_min_speed = 0.5,      # min EKF speed (px/frame) to use vel score
                                            # if EKF speed < this → vel_score = 0.5 (neutral)

            # ── NEW: Tiny / long-distance object ROI parameters ───────────────────
            # Used automatically when _is_long_distance() returns True.
            long_distance_area_fraction =  0.030,
            tiny_roi_start_expand            = 8.0,
            tiny_yolo_search_expand          = 20.0,
            tiny_search_expand_growth_factor = 1.3,
            tiny_search_expand_growth_every  = 50,
            tiny_search_expand_max           = 40.0,

            # ── DRM (existing params) ─────────────────────────────────────────────
            drm_tau_sim         = 0.6,
            mem_capacity        = 50,
            drm_mmin            = 3,
            drm_capacity        = 20,
            history_decay       = 0.1,
            drm_lam_dist        = 0.0,
            drm_lam_cand_dir    = 0.0,
            drm_lam_time        = 0.0,
            drm_lam_app         = 1,
            drm_lam_iou         = 0.0,
            drm_lam_mot         = 0.0,
            drm_margin          = 0.6,
            drm_skip_threshold  = 23,
            drm_top_k           = 100,
            vel_dir_hard_gate = 0.4,   # |cos| threshold below which score → 0.05
            yolo_filter_class  = False, # filter candidates to target class
            yolo_class_detect_frames   = 5,     # stride 

            # ── History ───────────────────────────────────────────────────────────
            conf_history_len    = 200,
            size_history_len    = 200,
            history_skip_last   = 8,

            # ── ROI expansion (normal objects) ────────────────────────────────────
            roi_start_expand                 = 20,
            yolo_search_expand               = 100,
            search_expand_growth_factor      = 1.4,
            search_expand_growth_every       = 150,
            search_expand_max                = 500.0,

            # ── Misc ──────────────────────────────────────────────────────────────
            velocity_window_average          = 200,
            shrinkage_max_lookback           = 30,
            enter_occlusion_on_loss          = True,
            drm_gamma                        = 0,


        )
    print(f"Processing video {i + 1}/{len(test_public_lb)}: {value['video_path']}")
    video_path = os.path.join(
        data_dir, value["video_path"]
    )
    ann_path = os.path.join(
        data_dir, value["annotation_path"]
    )

    output_path = os.path.join(
        outputs_dir, value["video_path"]
    )

    init_bbox = np.loadtxt(ann_path, delimiter=",", dtype=np.float32).tolist()

    run_inference(
        video_path=video_path,
        initial_bbox=init_bbox,
        tracker=tracker,
        output_path=output_path
    )

with open("/home/moha/AIC-4/data_competition/metadata/contestant_manifest.json", "r") as f:
    test_public_lb = json.load(f)["public_lb"]
submission_df = defaultdict(list)
for key, value in test_public_lb.items():
    video_path = os.path.join(data_dir, value["video_path"])
    output_path = os.path.join(outputs_dir, value["video_path"])
    print(video_path, output_path)

    head, tail = os.path.split(output_path)
    bbox_dir = os.path.join(head, 'bboxes')
    bbox_file = os.path.join(bbox_dir, os.path.splitext(tail)[0] + '.txt')

    seq_id = os.path.splitext(value["video_path"])[0]
    if os.path.exists(bbox_file):
        with open(bbox_file, 'r') as f:
            lines = f.read().strip().split('\n')

            for frame_idx, line in enumerate(lines):
                x, y, w, h = line.strip().split()
                submission_df["id"].append(f"{key}_{frame_idx}")
                submission_df["x"].append(float(x))
                submission_df["y"].append(float(y))
                submission_df["w"].append(float(w))
                submission_df["h"].append(float(h))

submission = pd.DataFrame(submission_df)
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"Total rows: {len(submission)}")

{'penalty_k': 0.062, 'window_influence': 0.38, 'lr': 0.4, 'windowing': 'cosine', 'total_stride': 16, 'score_size': 16, 'N': 10, 'dynamic_update': True, 'similarity_score': False, 'stride': 2, 'smooth': False, 'bbox_ratio': 0.5, 'template_bbox_offset': 0.2, 'search_context': 2, 'instance_size': 256, 'template_size': 128, 'memory_window_size': 20, 'dynamic_update_threshold': 0.87, 'running_confidence_floor_value': 100, 'iou_threshold': 0.6, 'warmup_frames': 35, 'warmup_window_size': 8}
Processing video 1/89: dataset1/Car_video/Car_video.mp4

─── Latency Report ───────────────────────────────────────
ALL FRAMES            n= 584  mean=16.8ms  med=14.4ms  p95=34.4ms  p99=39.8ms  min=12.2ms  max=52.2ms  fps=59.7
NORMAL TRACK          n= 584  mean=16.8ms  med=14.4ms  p95=34.4ms  p99=39.8ms  min=12.2ms  max=52.2ms  fps=59.7
OCCLUSION             no data
──────────────────────────────────────────────────────────

{'penalty_k': 0.062, 'window_influence': 0.38, 'lr': 0.4, 'windowing': 'cosine', 

In [ ]:
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"Total rows: {len(submission)}")

                     id      x      y      w      h
0  dataset1/Car_video_0  536.0  551.0  226.0  142.0
1  dataset1/Car_video_1  533.0  549.0  237.0  150.0
2  dataset1/Car_video_2  531.0  549.0  237.0  152.0
3  dataset1/Car_video_3  531.0  549.0  237.0  152.0
4  dataset1/Car_video_4  529.0  550.0  239.0  155.0
Total rows: 74293
